## Meeting Minutes Bot with Hybrid Memory System

<details><summary style="display:list-item; font-size:16px; color:blue;">Jupyter Help</summary>
    
Having trouble testing your work? Double-check that you have followed the steps below to write, run, save, and test your code!
    
[Click here for a walkthrough GIF of the steps below](https://static-assets.codecademy.com/Courses/ds-python/jupyter-help.gif)

Run all initial cells to import libraries and datasets. Then follow these steps for each question:
    
1. Add your solution to the cell with `## YOUR SOLUTION HERE ## `.
2. Run the cell by selecting the `Run` button or the `Shift`+`Enter` keys.
3. Save your work by selecting the `Save` button, the `command`+`s` keys (Mac), or `control`+`s` keys (Windows).
4. Select the `Test Work` button at the bottom left to test your work.

![Screenshot of the buttons at the top of a Jupyter Notebook. The Run and Save buttons are highlighted](https://static-assets.codecademy.com/Paths/ds-python/jupyter-buttons.png)

#### Setup

Run the cell below to import the necessary libraries and initialize the language model.

**Note:** The OpenAI API key is already configured in this workspace, so no additional setup is needed.

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
import tiktoken
import json

import os
os.environ["OPENAI_API_KEY"] = os.environ["OPENAI_API_KEY"]


# Initialize the LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

# Initialize token counter
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

In this project, you'll build a Meeting Minutes Bot that can participate in team meetings and maintain three types of memory:

1. **Buffer Memory**: Store the complete meeting transcript
2. **Summary Memory**: Generate a condensed overview of the meeting
3. **Entity Memory**: Extract and store key information (goals, decisions, action items)

You'll simulate a real Q1 Marketing Planning meeting with multiple participants and see how each memory system captures different aspects of the conversation.

### Task Group 1 - Buffer Memory System

**Task 1: Setting Up Buffer Memory Storage**

An empty dictionary named `meeting_store` is provided to hold conversation histories, allowing separate tracking of multiple meetings without mixing their contexts.

A function `get_meeting_history` is already defined that takes a `meeting_id` parameter to retrieve or initialize the conversation history for a specific meeting. 

Inside the function:

1. Check if the `meeting_id` exists in `meeting_store`; reuse it if yes, otherwise create and store a new `InMemoryChatMessageHistory` object.

3. Return the newly stored history object so it can be used to maintain conversation continuity.

In [ ]:
# Dictionary to store meeting conversation history
meeting_store = {}

def get_meeting_history(meeting_id: str):
    """Retrieve or create conversation history for a meeting."""
    if meeting_id not in meeting_store:
        meeting_store[meeting_id] = InMemoryChatMessageHistory()
    return meeting_store[meeting_id]

**Task 2: Creating the Silent Note-Taking Prompt**

1. Create a `ChatPromptTemplate` named `meeting_prompt` that includes a list with three elements so the bot acts as a silent note-taker rather than an active participant:
   - A `system` message tuple defining the bot as a silent AI note-taking assistant that only responds with a checkmark symbol.
   - A `MessagesPlaceholder` with variable name set to `history` to inject past conversation context.
   - A `human` message tuple with an `input` placeholder to receive new messages.

In [ ]:
# Create prompt template for meeting conversation
meeting_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a silent AI note-taking assistant for team meetings. Your role is to listen and record discussions between team members. Do not participate in conversations. Respond only with: ✓"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

**Task 3: Setting Up the Conversation Chain**

1. Create a `base_chain` by connecting `meeting_prompt`, `llm`, and an output parser so each component passes its output to the next.

2. Set up the `conversation_bot` by using `RunnableWithMessageHistory` to wrap the base chain, and provide the following parameters to link the chain with the message history storage system.
    - `base_chain` to define the core LLM pipeline
   - `get_meeting_history` to retrieve the correct message history
   - `input_messages_key` set to `"input"`
   - `history_messages_key` set to `"history"`

In [ ]:
# Create the base chain
base_chain = meeting_prompt | llm | StrOutputParser()

# Wrap chain with message history
conversation_bot = RunnableWithMessageHistory(
    base_chain,
    get_meeting_history,
    input_messages_key="input",
    history_messages_key="history"
)

**Task 4: Adding Messages to the Meeting**

A function `add_meeting_message` that enables recording speaker messages to the meeting transcript with proper identification and session management is already defined.

Inside the function:

1. Define `formatted_message`  that combines the `speaker_name` and `message` with a colon (`:`) separator to clearly identify who said what in the meeting transcript.

2. Define a `config` dictionary with a nested structure containing a `configurable` key that maps to another dictionary with the `session_id` to specify which meeting this message belongs to.

3. Call the invoke method on `conversation_bot`, passing a dictionary with the `input` key mapped to the `formatted_message` along with the `config` parameter, storing the result in `response` and returning it to process the message through the bot.

After implementing, we verify buffer memory by simulating the first exchanges of the Q1 Marketing Planning meeting.

In [ ]:
def add_meeting_message(meeting_id: str, speaker_name: str, message: str):
    """Add a message to the meeting and get bot response."""
    formatted_message = f"{speaker_name}: {message}"
    config = {"configurable": {"session_id": meeting_id}}
    
    response = conversation_bot.invoke({"input": formatted_message}, config=config)
    return response

# Test the buffer memory
print("=== Q1 Marketing Planning Meeting ===")
print("Participants: Sarah (Manager), John (Designer), Mike (Developer), Lisa (Marketing)")
print("Date: Monday, January 20, 2025")
print("Bot: [Silent note-taker mode - Recording meeting]\n")

# Turn 1
msg = "Good morning everyone. Let's discuss our Q1 marketing campaign. We need to increase social media engagement by 40%."
print(f"Sarah: {msg}")
response = add_meeting_message("Q1-Planning", "Sarah", msg)
print(f"[Bot recording: {response}]\n")

# Turn 2
msg = "I think we should redesign the Instagram templates. The current ones are outdated."
print(f"John: {msg}")
response = add_meeting_message("Q1-Planning", "John", msg)
print(f"[Bot recording: {response}]\n")

### Task Group 2 - Entity Memory System

**Task 5: Building Entity Memory Storage**

An empty dictionary named `entity_store` is provided to hold extracted structured information, such as goals, decisions, action items, and dates, separately from raw meeting transcripts.

A function `get_entities` is already defined that takes a `meeting_id` parameter to retrieve or initialize the entity storage for a meeting. 

Inside the function:

1.  Check if the `meeting_id` exists in the `entity_store`.

2. If not present, initialize the store entry as a dictionary with four keys `goals`, `decisions`, `action_items`, and `key_dates`, each mapping to an empty list to create separate storage buckets for different types of information.

3. Return the stored entity dictionary to provide access to the meeting's entity data.

In [ ]:
# Dictionary to store extracted entities
entity_store = {}

def get_entities(meeting_id: str):
    """Retrieve or create entity storage for a meeting."""
    if meeting_id not in entity_store:
        entity_store[meeting_id] = {
            "goals": [],
            "decisions": [],
            "action_items": [],
            "key_dates": []
        }
    return entity_store[meeting_id]

**Task 6: Adding Entities to Storage**

A function `add_entity` with parameters `meeting_id`, `entity_type`, and `entity_value` is already defined to add new extracted information to the appropriate category while preventing duplicates. 

Inside the function:

1. Call the `get_entities` function with the `meeting_id` and store the result in the variable `entities` to access the meeting's entity storage.

2. Check if `entity_value` is not already in the list at the `entity_type` key to prevent duplicates.

3. If not present, add `entity_value` to the appropriate list to add the new information.

In [ ]:
def add_entity(meeting_id: str, entity_type: str, entity_value: str):
    """Add an entity to the meeting's entity store."""
    entities = get_entities(meeting_id)
    if entity_value not in entities[entity_type]:
        entities[entity_type].append(entity_value)

A helper function `parse_json_response` is provided below to clean and parse JSON responses from the LLM. It handles cases where the LLM wraps JSON output in markdown code blocks (\```json or \```), removes these wrappers, and returns a clean Python dictionary. This function is used in entity extraction tasks to ensure reliable parsing of structured data from LLM responses.

In [ ]:
def parse_json_response(response) -> dict:
    """
    Parse JSON from LLM response, removing markdown code blocks if present.
    """
    content = response.content.strip()
    
    # Remove markdown code blocks if present
    if content.startswith("```json"):
        content = content.replace("```json", "").replace("```", "").strip()
    elif content.startswith("```"):
        content = content.replace("```", "").strip()
    
    return json.loads(content)

**Task 7: Extracting Entities from Meeting Messages**

The function `extract_entities_from_text` is already defined with an extraction prompt and error handling to extract key information such as goals, decisions, action items, and dates from individual meeting messages.

Inside the `try` block:

1. Invoke the `llm` with `extraction_prompt` and store in `response` to get the LLM's analysis of the message.

2. Call the `parse_json_response` function with the `response` object to clean and parse the JSON output from the LLM, then assign it to the `result` variable.

3. Loop through each `entity_type` in a list of the four entity categories `goals`, `decisions`, `action_items`, and `key_dates`. For each item found in the `result` dictionary for that type, call the `add_entity` function with the `meeting_id`, `entity_type`, and `item` value to store all extracted information in the entity store.

The `except` clauses catch `JSONDecodeError` and other exceptions and intentionally do nothing, allowing the function to continue without storing any entities for that message.

In [ ]:
def extract_entities_from_text(meeting_id: str, text: str):
    """Extract entities from meeting text using LLM."""
    extraction_prompt = f"""Analyze this meeting statement and extract key information.

Statement: "{text}"

Identify:
- Goals: What the team wants to achieve
- Decisions: What was decided or agreed upon
- Action Items: Tasks assigned (format: "Person: Task - Due: Date")
- Key Dates: Important deadlines or meeting dates

Respond with ONLY valid JSON in this exact format:
{{"goals": [], "decisions": [], "action_items": [], "key_dates": []}}

If nothing found for a category, use empty array."""
    
    try:
        response = llm.invoke(extraction_prompt)
                
        result = parse_json_response(response)
        
        # Iterate over each entity category and store extracted values
        for entity_type in ["goals", "decisions", "action_items", "key_dates"]:
            for item in result.get(entity_type, []):
                add_entity(meeting_id, entity_type, item)
                
    except json.JSONDecodeError:
        pass 
    except Exception:
        pass 

**Task 8: Building the Full Meeting Transcript**

A function named `build_full_meeting_transcript` is already defined to extract all speaker messages from the meeting history and combine them into a single transcript string. 

Inside the function:

1. Call `get_meeting_history` with the `meeting_id` and store it in `history` to access the conversation messages.

2. Create an empty list named `conversation_text` to collect all speaker messages.

3. Loop through each message in the `history`:
   - Check if the message type indicates a human speaker (rather than the bot).
   - If so, add the message content to `conversation_text` to gather only the actual meeting dialogue.

4. Join all collected messages from `conversation_text` into a single string and return the result to create a complete meeting transcript for analysis.

In [ ]:
def build_full_meeting_transcript(meeting_id: str):
    """Build complete meeting transcript from conversation history."""
    history = get_meeting_history(meeting_id)
    
    conversation_text = []
    for message in history.messages:
        if message.type == "human":
            conversation_text.append(message.content)
    
    return "\n".join(conversation_text)

**Task 9: Comprehensive Entity Extraction from Full Meeting**

A function named `extract_all_entities_from_meeting` is already defined to analyze the complete meeting transcript and extract all key information with full context. 

Inside the function:

1. Call `build_full_meeting_transcript` with the `meeting_id` and store it in `full_transcript` to get the complete meeting dialogue.

2. Inside the `try` block, invoke the `llm` with `extraction_prompt` and store in `response` to analyze the complete transcript.

3. The `parse_json_response` function cleans and parses the JSON output from the LLM, and assigns it to the `result` variable.

4. Get entities using `get_entities` with the `meeting_id` and store in `entities`, then loop through the `entity_type` and reset each to an empty list to clear any previous extractions before the comprehensive analysis.

5. Loop through `entity_type` again. For each item found in the `result` dictionary for that type, call `add_entity` to populate the `entity_store` with all extracted information.

Finally, we calculate total extracted entities by summing the lengths of all entity lists in the `result`, and print a confirmation message showing the total count.

In [ ]:
def extract_all_entities_from_meeting(meeting_id: str):
    """Extract entities from the complete meeting transcript."""
    full_transcript = build_full_meeting_transcript(meeting_id)
    
    extraction_prompt = f"""Analyze this complete meeting transcript and extract ALL key information.

Meeting Transcript: {full_transcript}

Identify:
- Goals: What the team wants to achieve (e.g., "Increase social media engagement by 40%")
- Decisions: What was decided or agreed upon (e.g., "Budget set at $50,000")
- Action Items: Tasks assigned with person and date (format: "Person: Task - Due: Specific Date")
- Key Dates: Important deadlines (use dates mentioned in the transcript)

Respond with ONLY valid JSON:
{{"goals": [], "decisions": [], "action_items": [], "key_dates": []}}

Extract ALL items. For dates, use the full format (e.g., "Friday, January 24, 2025")."""
    
    try:
        response = llm.invoke(extraction_prompt)
        
        result = parse_json_response(response)
        
        # Clear existing entities first
        entities = get_entities(meeting_id)
        for entity_type in ["goals", "decisions", "action_items", "key_dates"]:
            entities[entity_type] = []
        
        # Add all extracted entities
        for entity_type in ["goals", "decisions", "action_items", "key_dates"]:
            for item in result.get(entity_type, []):
                add_entity(meeting_id, entity_type, item)
        
        total_extracted = sum(len(v) for v in result.values())
        print(f"✓ Extracted {total_extracted} entities from meeting")
                
    except json.JSONDecodeError:
        print("⚠️  Entity extraction failed: Invalid JSON response from LLM")
    except Exception as e:
        print(f"⚠️  Entity extraction error: {str(e)}")

**Task 10: Displaying Extracted Entities**

A function named `display_entities` is already defined to enable viewing extracted entities in a readable format.

Inside the function:

1. Call the `get_entities` function with the `meeting_id` and store it in `entities` to access the extracted information.

2. Inside the loop, get the items for the current `entity_type` from the dictionary `entities` to retrieve the list of extracted items.

3. Check if the `items` list is empty, and if so, print that none were recorded. Otherwise, enumerate through the items and print each item with a number to display all extracted entities.

In [ ]:
def display_entities(meeting_id: str):
    """Display all extracted entities for a meeting."""
    entities = get_entities(meeting_id)
    
    print("=" * 60)
    print("EXTRACTED ENTITIES")
    print("=" * 60)
    
    for entity_type in ["goals", "decisions", "action_items", "key_dates"]:
        print(f"\n{entity_type.replace('_', ' ').title()}:")
        items = entities[entity_type]
        if not items:
            print("  None recorded")
        else:
            for i, item in enumerate(items, 1):
                print(f"  {i}. {item}")

**Task 11: Integrating Entity Extraction into Meeting Flow**

A function named `meeting_with_extraction` is already defined to integrate entity extraction into the meeting flow.

Inside the function:

1. Call the `extract_entities_from_text` function with the `meeting_id` and an f-string combining the `speaker_name` and `message` with a colon (`:`) separator to analyze the message for key information before storing it.

2. Call the `add_meeting_message` function with the `meeting_id`, `speaker_name`, and `message`, and store the result in `response`.

3. Return `response` to record the message in the meeting transcript and provide the bot’s acknowledgement.

After this, the meeting continues, and entity extraction is tested using additional messages.

In [ ]:
def meeting_with_extraction(meeting_id: str, speaker_name: str, message: str):
    """Add message to meeting and extract entities."""
    extract_entities_from_text(meeting_id, f"{speaker_name}: {message}")
    
    response = add_meeting_message(meeting_id, speaker_name, message)
    return response

# Test entity extraction
print("=== Continuing Q1 Marketing Planning Meeting ===")

# Turn 3
msg = "Great idea, John. Can you create 5 new template mockups by Friday, January 24th?"
print(f"Sarah: {msg}")
response = meeting_with_extraction("Q1-Planning", "Sarah", msg)
print(f"[Bot recording: {response}]\n")

# Turn 4
msg = "Sure Sarah, I can deliver those by January 24th. I'll make sure they align with our 40% engagement goal."
print(f"John: {msg}")
response = meeting_with_extraction("Q1-Planning", "John", msg)
print(f"[Bot recording: {response}]\n")

### Task Group 3 - Summary Memory System

A helper function `count_tokens` is provided below to measure the size of meeting transcripts. It loops through all messages in a meeting's history, encodes each message into tokens using the `tiktoken` library, and returns the total count. This will be used later in the report generation to show the efficiency difference between buffer memory (stores everything) and summary memory (condensed version).

In [ ]:
def count_tokens(meeting_id: str) -> int:
    """Count total tokens in meeting conversation."""
    history = get_meeting_history(meeting_id)
    total_tokens = 0
    
    for message in history.messages:
        tokens = encoding.encode(message.content)
        total_tokens += len(tokens)
    
    return total_tokens

**Task 12: Generating Meeting Summary**

A `generate_summary` function that creates a concise 3–4 sentence summary of the meeting, so users can quickly understand what was discussed without reading the full transcript, is already defined. 

Inside the function:

1. Call `build_full_meeting_transcript` with the `meeting_id` and store it in `full_transcript` to get the complete meeting dialogue.

2. A `summary_prompt` is defined that instructs the LLM to use the `full_transcript` and summarize the meeting in 3-4 sentences, focusing on key topics, decisions, and action items. 

3. Invoke the `llm` with `summary_prompt`, and store the result in the  `response` variable.

4. Return the generated summary content from the LLM's `response`.

In [ ]:
def generate_summary(meeting_id: str):
    """Generate a concise summary of the meeting."""
    full_transcript = build_full_meeting_transcript(meeting_id)

    summary_prompt = f"""Summarize this meeting in 3-4 sentences. Focus on key topics, decisions, and action items.
    Meeting Transcript: {full_transcript}
    Summary:"""
    
    response = llm.invoke(summary_prompt)
    return response.content

### Task Group 4 - Hybrid System Integration & Final Report

**Task 13: Displaying Meeting Transcript**

**Displaying Meeting Transcript**

A function named `display_transcript` is already defined to enable viewing the complete meeting transcript with clear speaker identification. 

Inside the  function:

1. Call `get_meeting_history` with the `meeting_id` and store it in `history` to access the conversation messages.

2. Enumerate through messages to get both the index and the message with line numbering.

3. Check the message type:
   - If it's a human speaker, print the index and message content.
   - Otherwise, print the index with `"[Bot]"` prefix and the message content to display the full transcript with clear speaker identification.

In [ ]:
def display_transcript(meeting_id: str):
    """Display the full meeting transcript."""
    history = get_meeting_history(meeting_id)
    
    print("FULL MEETING TRANSCRIPT")
    print("─" * 60)
    
    for i, message in enumerate(history.messages, 1):
        if message.type == "human":
            print(f"{i}. {message.content}")
        else:
            print(f"{i}. [Bot] {message.content}")

**Task 14: Generating a Comprehensive Meeting Report**

A function named `generate_meeting_report` is already defined to compile all three memory types (buffer, summary, entities) into one comprehensive report.

Inside the function:

1. Call `count_tokens` with the meeting ID and store in `transcript_tokens` to measure the full transcript size.

2. Call `generate_summary` with the meeting ID and store it in `summary` to get the condensed version.

3. The summary is encoded using the encoding object, and its length is stored in `summary_tokens` to measure compression efficiency.

4. Call `get_entities` with the meeting ID and store in `entities` to retrieve the extracted structured information.

5. Create a dictionary `report` with the following keys: `meeting_id`, `transcript_tokens`, `summary`, `summary_tokens`, and `entities` to compile all three memory types into one comprehensive report.

6. Return the `report` variable to provide the complete meeting analysis.

In [ ]:
def generate_meeting_report(meeting_id: str):
    """Generate a comprehensive 3-section meeting report."""
    transcript_tokens = count_tokens(meeting_id)
    
    summary = generate_summary(meeting_id)
    
    summary_tokens = len(encoding.encode(summary))
    
    entities = get_entities(meeting_id)
    
    report = {
        "meeting_id": meeting_id,
        "transcript_tokens": transcript_tokens,
        "summary": summary,
        "summary_tokens": summary_tokens,
        "entities": entities
    }
    return report

**Task 15: Running Complete Meeting Simulation**

The function `run_complete_meeting` is already defined to orchestrate the full meeting by processing all remaining messages and generating the final comprehensive report.

The loop iterates through the messages, unpacking each tuple to extract the speaker name and message, then printing them in a format that displays the conversation flow.

1. Call `meeting_with_extraction` with the `meeting_id`, `speaker`, and `message`, storing the result in `response` to process each message through both conversation storage and entity extraction.

After processing all messages:

2. Call `extract_all_entities_from_meeting` with the `meeting_id` to perform comprehensive entity extraction using the full meeting context.

3. Call `generate_meeting_report` with the `meeting_id` and return the result to compile conversation memory, summary memory, and entity memory into a single final report.

4. Finally, call the `run_complete_meeting` function with the meeting ID `"Q1-Planning"` and store the returned value in `report` to produce the complete meeting output.

In [ ]:
def run_complete_meeting(meeting_id: str):
    """Complete the meeting simulation with remaining turns."""
    remaining_messages = [
        ("Mike", "John, once you finish those templates, can you share them with me? I'm updating our landing page and want to keep the design consistent."),
        ("John", "Absolutely Mike. I'll send them over by end of day Friday, January 24th."),
        ("Mike", "Perfect. I'll start the landing page development on Monday, January 27th then. Should have a draft ready by end of that week."),
        ("Lisa", "Sarah, before we finalize designs, what's our total budget for this Q1 campaign? I need to know how much we can allocate for Instagram ads."),
        ("Sarah", "Good question Lisa. We have $50,000 for the entire Q1 campaign. Can you create a detailed budget breakdown by category?"),
        ("Lisa", "Yes, I'll have that ready by tomorrow afternoon, January 21st. I'll include Instagram ads, influencer partnerships, and content creation costs."),
        ("Sarah", "Excellent. So to recap: John delivers templates January 24th, Mike starts development January 27th, and Lisa provides budget breakdown January 21st."),
        ("Mike", "Just to confirm - I'll need John's templates before I can finalize the landing page design. Is that timeline okay?"),
        ("John", "Yes, you'll have them Friday, so plenty of time for your Monday start."),
        ("Sarah", "Perfect! Let's reconvene on Tuesday, January 28th to review everyone's progress. Meeting adjourned, thank you all.")
    ]
    
    for speaker, message in remaining_messages:
        print(f"{speaker}: {message}")
        response = meeting_with_extraction(meeting_id, speaker, message)
        print(f"[Bot recording: {response}]\n")
    
    print("\n" + "="*60)
    print("Extracting entities from complete meeting...")
    print("="*60 + "\n")
    # Extract all entities from complete meeting at the end
    extract_all_entities_from_meeting(meeting_id)
    
    return generate_meeting_report(meeting_id)

# Run the complete meeting
print("\n" + "="*60)
print("Completing the Q1 Marketing Planning Meeting")
print("="*60 + "\n")

report = run_complete_meeting("Q1-Planning")

#### Final Report Display

Now let's display the complete 3-section meeting report. Run the cell below to see all three memory systems in action!

In [ ]:
print("\n" + "="*70)
print("                    MEETING REPORT")
print("              Q1 Marketing Planning Meeting")
print("="*70)
print()

print(f"Meeting ID: {report['meeting_id']}")
print(f"Participants: Sarah (Manager), John (Designer), Mike (Developer), Lisa (Marketing)")
print()

# Section 1: Buffer Memory
print("─" * 70)
print("SECTION 1: BUFFER MEMORY (Full Transcript)")
print("─" * 70)
print(f"Token Count: {report['transcript_tokens']:,} tokens\n")
display_transcript(report['meeting_id'])
print()

# Section 2: Summary Memory
print("─" * 70)
print("SECTION 2: SUMMARY MEMORY (Meeting Overview)")
print("─" * 70)
print(f"Token Count: {report['summary_tokens']:,} tokens")
reduction = ((report['transcript_tokens'] - report['summary_tokens']) / report['transcript_tokens'] * 100)
print(f"Token Reduction: {reduction:.1f}%\n")
print(report['summary'])
print()

# Section 3: Entity Memory
print("─" * 70)
print("SECTION 3: ENTITY MEMORY (Key Information Extracted)")
print("─" * 70)
display_entities(report['meeting_id'])
print()
print("="*70)

#### Memory System Analysis

Let's analyze the effectiveness of each memory type. Run the cell below to see the comparison!

In [ ]:
print("="*70)
print("MEMORY SYSTEM ANALYSIS")
print("="*70)
print()

print("1. BUFFER MEMORY (Full Transcript)")
print("   Purpose: Complete, verbatim record of all conversation")
print(f"   Size: {report['transcript_tokens']:,} tokens")
print("   Use Cases:")
print("   - Exact quote retrieval")
print("   - Dispute resolution")
print("   - Legal/compliance records")
print("   - Full context understanding")
print()

print("2. SUMMARY MEMORY (Condensed Overview)")
print("   Purpose: Quick understanding without reading full transcript")
print(f"   Size: {report['summary_tokens']:,} tokens")
reduction = ((report['transcript_tokens'] - report['summary_tokens']) / report['transcript_tokens'] * 100)
print(f"   Efficiency: {reduction:.1f}% token reduction")
print("   Use Cases:")
print("   - Quick catch-up for absent members")
print("   - Executive briefings")
print("   - Email recaps")
print("   - Status reports")
print()

print("3. ENTITY MEMORY (Structured Data)")
entities = report['entities']
total_entities = sum(len(v) for v in entities.values())
print("   Purpose: Queryable, structured information")
print(f"   Size: {total_entities} entities extracted")
print(f"   - Goals: {len(entities['goals'])}")
print(f"   - Decisions: {len(entities['decisions'])}")
print(f"   - Action Items: {len(entities['action_items'])}")
print(f"   - Key Dates: {len(entities['key_dates'])}")
print("   Use Cases:")
print("   - Task tracking systems")
print("   - Calendar integration")
print("   - Progress dashboards")
print("   - Specific queries ('What are John's tasks?')")
print()

print("HYBRID MEMORY ADVANTAGE:")
print("By combining all three memory types, the bot provides:")
print("✓ Complete accuracy (Buffer Memory)")
print("✓ Quick access (Summary Memory)")
print("✓ Actionable data (Entity Memory)")
print()
print("="*70)